In [0]:
from pyspark.sql.functions import *

df = (spark.read
      .format("csv")
      .option("header", True)
      .option("inferSchema", True)
      .load("/Volumes/workspace/default/pyspark_files/BigMart Sales.csv"))

df.display()

## User-Defined Functions (UDFs)

UDFs allow the use of custom Python logic within Spark.

* **Process:** Define a Python function -&gt; Register it with `udf()` \-&gt; Apply to a column.
* **Performance Caveat:** UDFs are generally **slower**. Spark must spin up a Python interpreter inside the JVM (Java Virtual Machine) executors, leading to overhead. Native Spark functions are always preferred.

In [0]:
def myfun(x):
    return x*x

In [0]:
my_udf = udf(myfun)

In [0]:
df.withColumn('new_result', my_udf(col('Item_MRP'))).limit(5).display()

## Data Persistence: Writing Data

Writing data involves selecting a format and a **Mode**.

| **Mode**      | **Effect**                                           |
| ------------- | ---------------------------------------------------- |
| **Append**    | Adds new files to the existing folder.               |
| **Overwrite** | Deletes existing data in the folder and replaces it. |
| **Error**     | Throws an error if data already exists (default).    |
| **Ignore**    | Does nothing if data already exists.                 |

In [0]:
df.write.format("csv") \
    .save("/Volumes/workspace/default/pyspark_files/output")

#df.write.csv("/Volumes/workspace/default/pyspark_files/output")

In [0]:
#Existing data + new data
df.write.format("csv") \
    .mode("append") \
    .save("/Volumes/workspace/default/pyspark_files/output")

In [0]:
#Delete/replace existing output → write new data
df.write.format("csv") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/pyspark_files/output")

In [0]:
df.write.format("csv") \
    .mode("error") \
    .save("/Volumes/workspace/default/pyspark_files/output")

#or .mode("errorifexists")

In [0]:
df.write \
    .format("csv") \
    .mode("ignore") \
    .save("/Volumes/workspace/default/pyspark_files/output")

####Parquet and Table format
Parquet = a file/data format, like CSV. Parquet is commonly used in Spark because it's much more efficient for analytics than CSV. Parquet stores the data in a way optimized for reading particular columns.

Table = a Spark/Databricks object that you can query with SQL. Save this DataFrame as a table that I can query using SQL

In [0]:
df.write \
    .format("parquet") \
    .save("/Volumes/workspace/default/pyspark_files/data.parquet")

In [0]:
#creates a managed table

df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("my_table")

### Managed vs External Tables

* **Managed Tables:** Spark manages both the metadata and the actual data files. If you `DROP` the table, the data is deleted.
* **External Tables:** You provide a specific `location`. If you `DROP` the table, only the metadata is deleted; the physical data files remain safe.

## PySpark SQL

Spark allows you to run standard SQL queries on DataFrames.

* **Requirement:** Convert the DataFrame to a view using `df.createOrReplaceTempView("view_name")`.
* **Execution:** Use `spark.sql("SELECT * FROM view_name")` to return a new DataFrame.

| **Interview Tip** | **Question:** Is there a performance difference between PySpark API and Spark SQL? **Answer:** No. Both are optimised by the Catalyst Optimizer into the same underlying execution plan. |
| ----------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |

In [0]:
df.createTempView('my_view')

In [0]:
%sql
select * from my_view where Outlet_Type == 'Grocery Store'

In [0]:
df_sql = spark.sql("select * from my_view where Item_Fat_Content = 'Low Fat'")

In [0]:
df_sql.display()